In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import time
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient
from rich import print as rprint
from huggingface_hub import login, HfApi, hf_hub_download

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(token = HF_TOKEN)
api = HfApi()

rprint(f"GPU Devices: {tf.config.list_physical_devices('GPU')}")

os.makedirs('/kaggle/working/data', exist_ok = True)
os.makedirs('/kaggle/working/processed', exist_ok = True)

GPU Devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), 
PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

In [2]:
ml25_url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
zip_path_25m = '/kaggle/working/data/ml-25m.zip'

if not os.path.exists(zip_path_25m):
    rprint("Downloading ml-25m...")
    urllib.request.urlretrieve(ml25_url, zip_path_25m)

with zipfile.ZipFile(zip_path_25m, 'r') as z:
    z.extractall('/kaggle/working/data')

ratings = pd.read_csv(
    '/kaggle/working/data/ml-25m/ratings.csv',
    dtype = {
        'userId': 'int32', 
        'movieId': 'int32', 
        'rating': 'float32'
    }
)

rprint(f"Ratings: {ratings.shape}")

Downloading ml-25m...

Ratings: (25000095, 4)

In [3]:
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
movie_to_idx = {m: i for i, m in enumerate(movie_ids)}

ratings['user_idx'] = ratings['userId'].map(user_to_idx).astype('int32')
ratings['movie_idx'] = ratings['movieId'].map(movie_to_idx).astype('int32')

n_users = len(user_ids)
n_movies = len(movie_ids)
rprint(f"n_users: {n_users}, n_movies: {n_movies}")

ratings['rating_scaled'] = (ratings['rating'] - 0.5) / 4.5 

rprint(ratings[['userId', 'user_idx', 'movieId', 'movie_idx', 'rating', 'rating_scaled']].head())

n_users: 162541, n_movies: 59047

userId  user_idx  movieId  movie_idx  rating  rating_scaled
0       1         0      296          0     5.0       1.000000
1       1         0      306          1     3.5       0.666667
2       1         0      307          2     5.0       1.000000
3       1         0      665          3     5.0       1.000000
4       1         0      899          4     3.5       0.666667

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(ratings, test_size = 0.1, random_state = 42)
rprint("Train:", train_df.shape, "| Test:", test_df.shape)

def make_dataset(df, batch_size = 8192, shuffle = True):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            {
                'user_idx': df['user_idx'].values, 
                'movie_idx': df['movie_idx'].values
            },
            df['rating_scaled'].values
        )
    )
    
    if shuffle:
        ds = ds.shuffle(buffer_size = 100_000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle = True)
test_ds = make_dataset(test_df, shuffle = False)

rprint("Dataset pipeline ready.")

Train:
(22500085, 7)
| Test:
(2500010, 7)

I0000 00:00:1785217929.003492      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785217929.006815      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Dataset pipeline ready.

In [5]:
embedding_dim = 32

user_input = tf.keras.Input(shape = (1,), name = 'user_idx')
movie_input = tf.keras.Input(shape = (1,), name = 'movie_idx')

user_embedding = tf.keras.layers.Embedding(n_users, embedding_dim, name = 'user_embedding')(user_input)
movie_embedding = tf.keras.layers.Embedding(n_movies, embedding_dim, name = 'movie_embedding')(movie_input)

user_vec = tf.keras.layers.Flatten()(user_embedding)
movie_vec = tf.keras.layers.Flatten()(movie_embedding)

concat = tf.keras.layers.Concatenate()([user_vec, movie_vec])

x = tf.keras.layers.Dense(128, activation = 'relu')(concat)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(64, activation = 'relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(32, activation = 'relu')(x)
output = tf.keras.layers.Dense(1, activation = 'sigmoid', name = 'rating')(x)

ncf_model = tf.keras.Model(inputs = [user_input, movie_input], outputs = output)
ncf_model.compile(optimizer = 'adam', loss = 'mse', metrics = ['mae'])
ncf_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_idx           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │  5,201,312 │ user_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 1, 32)     │  1,889,504 │ movie_idx[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ movie_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      8,320 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rating (Dense)      │ (None, 1)         │         33 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,109,505 (27.12 MB)

 Trainable params: 7,109,505 (27.12 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
sample_train = train_df.sample(n = 1_000_000, random_state = 42)
sample_ds = make_dataset(sample_train, batch_size = 8192)

start = time.time()
history_test = ncf_model.fit(sample_ds, epochs = 1, verbose = 1)
elapsed = time.time() - start
rprint(f"\n1M rows, 1 epoch time: {elapsed:.1f} seconds")
rprint(f"Estimated time for full 22.5M train, 1 epoch: ~{elapsed * 22.5 / 60:.1f} minutes")

  5/123 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - loss: 0.0812 - mae: 0.2418

I0000 00:00:1785218117.200981     629 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


123/123 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - loss: 0.0494 - mae: 0.1730


1M rows, 1 epoch time: 13.3 seconds

Estimated time for full 22.5M train, 1 epoch: ~5.0 minutes

In [7]:
checkpoint_path = '/kaggle/working/processed/ncf_checkpoint.weights.h5'

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath = checkpoint_path,
    save_weights_only = True,
    save_best_only = True,
    monitor = 'val_loss',
    verbose = 1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 2,
    restore_best_weights = True
)

user_input = tf.keras.Input(shape = (1,), name = 'user_idx')
movie_input = tf.keras.Input(shape = (1,), name = 'movie_idx')
user_embedding = tf.keras.layers.Embedding(n_users, embedding_dim, name = 'user_embedding')(user_input)
movie_embedding = tf.keras.layers.Embedding(n_movies, embedding_dim, name = 'movie_embedding')(movie_input)
user_vec = tf.keras.layers.Flatten()(user_embedding)
movie_vec = tf.keras.layers.Flatten()(movie_embedding)
concat = tf.keras.layers.Concatenate()([user_vec, movie_vec])
x = tf.keras.layers.Dense(128, activation = 'relu')(concat)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(64, activation = 'relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(32, activation = 'relu')(x)
output = tf.keras.layers.Dense(1, activation = 'sigmoid', name = 'rating')(x)
ncf_model = tf.keras.Model(inputs = [user_input, movie_input], outputs = output)
ncf_model.compile(optimizer = 'adam', loss = 'mse', metrics = ['mae'])

start = time.time()

history = ncf_model.fit(
    train_ds,
    validation_data = test_ds,
    epochs = 10,
    callbacks = [checkpoint_callback, early_stop],
    verbose = 1
)

elapsed = time.time() - start

rprint(f"\nTotal training time: {elapsed/60:.1f} minutes")

Epoch 1/10
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 0.0405 - mae: 0.1543
Epoch 1: val_loss improved from None to 0.03480, saving model to /kaggle/working/processed/ncf_checkpoint.weights.h5

Epoch 1: finished saving model to /kaggle/working/processed/ncf_checkpoint.weights.h5
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 183s 65ms/step - loss: 0.0369 - mae: 0.1466 - val_loss: 0.0348 - val_mae: 0.1443
Epoch 2/10
2746/2747 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.0338 - mae: 0.1399
Epoch 2: val_loss improved from 0.03480 to 0.03369, saving model to /kaggle/working/processed/ncf_checkpoint.weights.h5

Epoch 2: finished saving model to /kaggle/working/processed/ncf_checkpoint.weights.h5
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 170s 62ms/step - loss: 0.0334 - mae: 0.1389 - val_loss: 0.0337 - val_mae: 0.1417
Epoch 3/10
2746/2747 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.0322 - mae: 0.1362
Epoch 3: val_loss improved from 0.03369 to 0.03283, saving model to /kaggle/working/processed/ncf_checkpoint.weig

Total training time: 25.4 minutes

In [8]:
ncf_model.load_weights('/kaggle/working/processed/ncf_checkpoint.weights.h5')

eval_result = ncf_model.evaluate(test_ds, verbose = 0)

rprint(f"Reloaded model - val_loss: {eval_result[0]:.5f}, val_mae: {eval_result[1]:.5f}")

Reloaded model - val_loss: 0.03203, val_mae: 0.13759

In [9]:
rmse_ncf_scaled = np.sqrt(eval_result[0])
rmse_ncf_original = rmse_ncf_scaled * 4.5 

mae_ncf_original = eval_result[1] * 4.5

rprint(f"NCF RMSE (original 0.5-5 scale): {rmse_ncf_original:.4f}")
rprint(f"NCF MAE (original 0.5-5 scale): {mae_ncf_original:.4f}")
rprint()
rprint(f"Comparison:")
rprint(f"  SVD (Stage 3): RMSE=0.7728, MAE=0.5832")
rprint(f"  NCF (Stage 4): RMSE={rmse_ncf_original:.4f}, MAE={mae_ncf_original:.4f}")

NCF RMSE (original 0.5-5 scale): 0.8054

NCF MAE (original 0.5-5 scale): 0.6192

Comparison:

SVD (Stage 3): RMSE=0.7728, MAE=0.5832

NCF (Stage 4): RMSE=0.8054, MAE=0.6192

In [10]:
ncf_model.load_weights('/kaggle/working/processed/ncf_checkpoint.weights.h5')
eval_result = ncf_model.evaluate(test_ds, verbose = 0)

rprint(f"Reloaded model - val_loss: {eval_result[0]:.5f}, val_mae: {eval_result[1]:.5f}")

rmse_ncf_scaled = np.sqrt(eval_result[0])
rmse_ncf_original = rmse_ncf_scaled * 4.5
mae_ncf_original = eval_result[1] * 4.5

rprint(f"\nNCF RMSE (original 0.5-5 scale): {rmse_ncf_original:.4f}")
rprint(f"NCF MAE (original 0.5-5 scale): {mae_ncf_original:.4f}")
rprint()
rprint(f"SVD (Stage 3):  RMSE = 0.7728, MAE = 0.5832")
rprint(f"NCF (Stage 4):  RMSE = {rmse_ncf_original:.4f}, MAE = {mae_ncf_original:.4f}")

Reloaded model - val_loss: 0.03203, val_mae: 0.13759

NCF RMSE (original 0.5-5 scale): 0.8054

NCF MAE (original 0.5-5 scale): 0.6192

SVD (Stage 3):  RMSE = 0.7728, MAE = 0.5832

NCF (Stage 4):  RMSE = 0.8054, MAE = 0.6192

In [11]:
embedding_dim_v2 = 64

user_input = tf.keras.Input(shape = (1,), name = 'user_idx')
movie_input = tf.keras.Input(shape = (1,), name = 'movie_idx')

user_embedding = tf.keras.layers.Embedding(n_users, embedding_dim_v2, name = 'user_embedding')(user_input)
movie_embedding = tf.keras.layers.Embedding(n_movies, embedding_dim_v2, name = 'movie_embedding')(movie_input)

user_vec = tf.keras.layers.Flatten()(user_embedding)
movie_vec = tf.keras.layers.Flatten()(movie_embedding)

concat = tf.keras.layers.Concatenate()([user_vec, movie_vec])

x = tf.keras.layers.Dense(256, activation = 'relu')(concat)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation = 'relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation = 'relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
output = tf.keras.layers.Dense(1, activation = 'sigmoid', name = 'rating')(x)

ncf_model_v2 = tf.keras.Model(inputs = [user_input, movie_input], outputs = output)
ncf_model_v2.compile(optimizer = 'adam', loss = 'mse', metrics = ['mae'])
ncf_model_v2.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_idx            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_idx           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 64)     │ 10,402,624 │ user_idx[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 1, 64)     │  3,779,008 │ movie_idx[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 64)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 64)        │          0 │ movie_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 128)       │          0 │ flatten_4[0][0],  │
│ (Concatenate)       │                   │            │ flatten_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256)       │     33,024 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128)       │     32,896 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rating (Dense)      │ (None, 1)         │         65 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,255,873 (54.38 MB)

 Trainable params: 14,255,873 (54.38 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
checkpoint_path_v2 = '/kaggle/working/processed/ncf_checkpoint_v2.weights.h5'

checkpoint_callback_v2 = tf.keras.callbacks.ModelCheckpoint(
    filepath = checkpoint_path_v2,
    save_weights_only = True,
    save_best_only = True,
    monitor = 'val_loss',
    verbose = 1
)

early_stop_v2 = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 3,
    restore_best_weights = True
)

start = time.time()

history_v2 = ncf_model_v2.fit(
    train_ds,
    validation_data = test_ds,
    epochs = 8,
    callbacks = [checkpoint_callback_v2, early_stop_v2],
    verbose = 1
)

elapsed = time.time() - start
rprint(f"\nTotal training time: {elapsed/60:.1f} minutes")

Epoch 1/8
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0405 - mae: 0.1543
Epoch 1: val_loss improved from None to 0.03428, saving model to /kaggle/working/processed/ncf_checkpoint_v2.weights.h5

Epoch 1: finished saving model to /kaggle/working/processed/ncf_checkpoint_v2.weights.h5
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 165s 58ms/step - loss: 0.0369 - mae: 0.1469 - val_loss: 0.0343 - val_mae: 0.1418
Epoch 2/8
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0338 - mae: 0.1401
Epoch 2: val_loss improved from 0.03428 to 0.03292, saving model to /kaggle/working/processed/ncf_checkpoint_v2.weights.h5

Epoch 2: finished saving model to /kaggle/working/processed/ncf_checkpoint_v2.weights.h5
2747/2747 ━━━━━━━━━━━━━━━━━━━━ 147s 54ms/step - loss: 0.0333 - mae: 0.1390 - val_loss: 0.0329 - val_mae: 0.1379
Epoch 3/8
2746/2747 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0319 - mae: 0.1356
Epoch 3: val_loss improved from 0.03292 to 0.03209, saving model to /kaggle/working/processed/ncf_checkp

Total training time: 19.7 minutes

In [13]:
ncf_model_v2.load_weights('/kaggle/working/processed/ncf_checkpoint_v2.weights.h5')
eval_result_v2 = ncf_model_v2.evaluate(test_ds, verbose = 0)

rprint(f"Reloaded v2 model - val_loss: {eval_result_v2[0]:.5f}, val_mae: {eval_result_v2[1]:.5f}")

rmse_ncf_v2 = np.sqrt(eval_result_v2[0]) * 4.5
mae_ncf_v2 = eval_result_v2[1] * 4.5

rprint(f"\nNCF v2 RMSE (original scale): {rmse_ncf_v2:.4f}")
rprint(f"NCF v2 MAE (original scale): {mae_ncf_v2:.4f}")
rprint()
rprint("Final comparison:")
rprint(f"  SVD (Stage 3):        RMSE = 0.7728, MAE = 0.5832")
rprint(f"  NCF v1 (32-dim):       RMSE = 0.8054, MAE = 0.6192")
rprint(f"  NCF v2 (64-dim, deeper): RMSE ={rmse_ncf_v2:.4f}, MAE = {mae_ncf_v2:.4f}")

Reloaded v2 model - val_loss: 0.03171, val_mae: 0.13401

NCF v2 RMSE (original scale): 0.8013

NCF v2 MAE (original scale): 0.6030

Final comparison:

SVD (Stage 3):        RMSE = 0.7728, MAE = 0.5832

NCF v1 (32-dim):       RMSE = 0.8054, MAE = 0.6192

NCF v2 (64-dim, deeper): RMSE =0.8013, MAE = 0.6030

In [14]:
import pickle

ncf_model_v2.save('/kaggle/working/processed/ncf_model_v2.keras')

api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/ncf_model_v2.keras',
    path_in_repo = 'ncf_model_v2.keras',
    repo_id = 'Subhadip007/UERP_Model',
    repo_type = 'model',
)

rprint("NCF v2 model pushed to HF Hub.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

NCF v2 model pushed to HF Hub.